# Webcam Discovery Colab Validation (Location-Agnostic)
Set `QUERY` and optionally `SAMPLE_STREAMS`. This notebook runs real discovery and audits artifacts.

In [ ]:
QUERY = "public live webcams near Eiffel Tower"
SAMPLE_STREAMS = []
BRANCH = "main"
REPO_URL = "https://github.com/dshipley71/webcam-discovery.git"


In [ ]:
import os
if not os.path.exists('webcam-discovery'):
    get_ipython().system('git clone {REPO_URL}')
get_ipython().run_line_magic('cd', 'webcam-discovery')
get_ipython().system('git fetch --all')
get_ipython().system('git checkout {BRANCH}')
get_ipython().system('git pull --ff-only')
get_ipython().system('python -m pip install -U pip')
get_ipython().system('python -m pip install -e .')


In [ ]:
import os
import certifi
from google.colab import userdata

ALLOW_INSECURE_SSL_FALLBACK = False

!apt-get update -qq
!apt-get install -y -qq ca-certificates
!update-ca-certificates
!python -m pip install -U certifi truststore

os.environ['WCD_CA_BUNDLE'] = certifi.where()
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
os.environ['CURL_CA_BUNDLE'] = certifi.where()

os.environ.setdefault('WCD_PLANNER_PROVIDER', 'ollama')
os.environ['WCD_OLLAMA_API_KEY'] = userdata.get('OLLAMA_API_KEY')
os.environ['WCD_OLLAMA_BASE_URL'] = userdata.get('WCD_OLLAMA_BASE_URL') or 'https://ollama.com'
if not os.environ.get('WCD_OLLAMA_API_KEY'):
    raise RuntimeError('Missing WCD_OLLAMA_API_KEY. Configure OLLAMA_API_KEY in Colab secrets before running agentic discovery.')

if ALLOW_INSECURE_SSL_FALLBACK:
    os.environ['WCD_ALLOW_INSECURE_SSL_FALLBACK'] = 'true'
else:
    os.environ.pop('WCD_ALLOW_INSECURE_SSL_FALLBACK', None)

print('WCD_CA_BUNDLE =', os.environ['WCD_CA_BUNDLE'])
print('ALLOW_INSECURE_SSL_FALLBACK =', ALLOW_INSECURE_SSL_FALLBACK)


In [ ]:
import subprocess, threading, time
from pathlib import Path

artifact_dir = Path('artifacts') / f'run_{int(time.time())}'
artifact_dir.mkdir(parents=True, exist_ok=True)

cmd=['webcam-discovery','run-agentic',QUERY,'--output-dir',str(artifact_dir),'--enable-feed-discovery','--max-feed-endpoints','100','--max-feed-records','3000','--max-stream-candidates','2500','--per-source-stream-cap','500','--preserve-direct-streams','--max-streams','500','--max-search-queries','25','--max-search-results-per-query','10','--llm-provider','ollama','--llm-model','gemma3:27b','--enable-visual-analysis']
print('Running:', ' '.join(cmd))
env=os.environ.copy()
p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
lines=[]
last=[time.time()]
def reader():
    for line in p.stdout:
        line=line.rstrip()
        lines.append(line)
        print(line)
        last[0]=time.time()
threading.Thread(target=reader,daemon=True).start()
while p.poll() is None:
    time.sleep(5)
    if time.time()-last[0] > 30:
        print('[heartbeat] process still running...')
print('Exit:', p.returncode)
print('Artifact dir:', artifact_dir)
funnel=artifact_dir/'logs'/'discovery_funnel.json'
print('discovery_funnel.json exists:', funnel.exists())
ssl_lines=[ln for ln in lines if 'SSL' in ln or 'certificate verify' in ln.lower()]
print('SSL-related lines:', len(ssl_lines))
for ln in ssl_lines[-10:]:
    print(ln)
camera_geojson=artifact_dir/'camera.geojson'
camera_md=artifact_dir/'cameras.md'
print('camera.geojson exists:', camera_geojson.exists())
print('cameras.md exists:', camera_md.exists())
if p.returncode != 0:
    raise RuntimeError('run-agentic failed')
